# DOWNLOADING DATASET TINYSTORIES

In [1]:
from huggingface_hub import snapshot_download

try:
    snapshot_download(
        repo_id="roneneldan/TinyStories",
        repo_type="dataset",
        local_dir="TinyStories"
    )
except Exception as e:
    import traceback
    traceback.print_exc()

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 14 files: 100%|██████████| 14/14 [01:01<00:00,  4.36s/it]


In [2]:
import glob
import os

for f in glob.glob("TinyStories/data/*.parquet"):
    print(os.path.basename(f), os.path.getsize(f) / 1024 / 1024, "MB")

train-00002-of-00004-a26307300439e943.parquet 234.50362586975098 MB
validation-00000-of-00001-869c898b519ad725.parquet 9.526373863220215 MB
train-00003-of-00004-d243063613e5a057.parquet 236.50012016296387 MB
train-00000-of-00004-2d5a1467fff1081b.parquet 237.2084722518921 MB
train-00001-of-00004-5852b56a2bd28fd9.parquet 236.67524337768555 MB


In [3]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="roneneldan/TinyStories",
    repo_type="dataset",
    local_dir="TinyStories",
    force_download=True
)

Fetching 14 files: 100%|██████████| 14/14 [12:26<00:00, 53.36s/it]  


'/workspace/TinyStories'

# Verification

In [4]:
import glob
import os

for f in glob.glob("TinyStories/data/*.parquet"):
    print(os.path.basename(f), os.path.getsize(f) / 1024 / 1024, "MB")

train-00002-of-00004-a26307300439e943.parquet 234.50362586975098 MB
validation-00000-of-00001-869c898b519ad725.parquet 9.526373863220215 MB
train-00003-of-00004-d243063613e5a057.parquet 236.50012016296387 MB
train-00000-of-00004-2d5a1467fff1081b.parquet 237.2084722518921 MB
train-00001-of-00004-5852b56a2bd28fd9.parquet 236.67524337768555 MB


# ==================================================
# DATA PREPARATION SCRIPT
# ==================================================

In [5]:
# This script prepares the TinyStories dataset for GPT training.
#
# It performs the following tasks:
# 1. Loads the TinyStories dataset
# 2. Converts stories into GPT-2 token IDs
# 3. Saves token IDs into binary (.bin) files
# 4. Saves tokenizer metadata for training
#
# Output Files:
#   train.bin
#   val.bin
#   meta.json
# ==================================================


# ==================================================
# IMPORT LIBRARIES
# ==================================================

# Provides operating system utilities such as
# creating directories and handling file paths.
import os

# Used for reading and writing JSON files.
import json

# Used to search for files using wildcard patterns.
# Example: train*.parquet
import glob

# Used for type hints.
# List[str] indicates a list containing strings.
from typing import List

# Numerical computing library.
# Used to store token IDs efficiently.
import numpy as np

# Pandas library.
# Used to read and combine Parquet files.
import pandas as pd

# GPT tokenizer library.
# Provides the same tokenizer used by GPT-2.
import tiktoken

# Displays a progress bar while processing data.
from tqdm import tqdm


# ==================================================
# CONFIGURATION
# ==================================================

# GPT-2 tokenizer name.
# Uses Byte Pair Encoding (BPE).
TOKENIZER_NAME = "gpt2"

# Folder containing the downloaded TinyStories
# Parquet files.
DATASET_PATH = "TinyStories/data"

# Folder where the processed binary files
# and metadata will be saved.
OUTPUT_DIR = "data/tinystories_tokenized"


# ==================================================
# TOKENIZATION FUNCTION
# ==================================================

# Converts a list of stories into GPT token IDs.
def encode_corpus(

    # List containing all stories.
    texts: List[str],

    # GPT-2 tokenizer object.
    enc: tiktoken.Encoding

# Function returns a NumPy array of token IDs.
) -> np.ndarray:

    """
    Encodes a list of stories into one continuous
    stream of GPT token IDs.
    """

    # Empty list that will store all token IDs.
    all_ids = []

    # Get the End-of-Text (EOT) token.
    # This token separates one story from the next.
    eot_token = enc.eot_token

    # Process every story one by one.
    for text in tqdm(

        # Input list of stories.
        texts,

        # Progress bar title.
        desc="Encoding texts"
    ):

        # Convert the current story into GPT token IDs.
        # encode_ordinary() is slightly faster than encode()
        # because it ignores special tokens.
        ids = enc.encode_ordinary(text)

        # Append all token IDs of the current story
        # to the master list.
        all_ids.extend(ids)

        # Append the End-of-Text token so the model
        # knows where one story ends.
        all_ids.append(eot_token)

    # Convert the complete list of token IDs
    # into a NumPy array.
    return np.array(

        # List of all token IDs.
        all_ids,

        # Store each token using 16 bits.
        # GPT-2 vocabulary (50,257 tokens) fits
        # comfortably within uint16.
        dtype=np.uint16
    )

# ==================================================
# MEMORY MAPPED FILE WRITER
# ==================================================
# Saves token IDs efficiently.
#
# Instead of loading everything into RAM,
# data can be read directly from disk.

def write_memmap(
    path: str,              # Path where the binary file will be saved
    tokens: np.ndarray      # NumPy array containing token IDs
):

    # Documentation string
    """Writes a numpy array of tokens to a memory-mapped file."""

    # Create a memory-mapped binary file.
    # mode="w+" creates a new file (or overwrites an existing one)
    # and allows both reading and writing.
    arr = np.memmap(
        path,               # Output file path
        dtype=np.uint16,    # Store each token as an unsigned 16-bit integer
        mode="w+",          # Create file for reading and writing
        shape=(tokens.size,)# Allocate enough space for all tokens
    )

    # Copy all token IDs into the memory-mapped file.
    arr[:] = tokens

    # Flush data from memory to disk.
    # Ensures all token IDs are permanently written.
    arr.flush()

    # Release the memory-mapped object.
    # This closes the file safely.
    del arr

    # Display a confirmation message.
    print(
        f"Wrote {tokens.size:,} tokens to {path}"
    )


In [6]:
# ==================================================
# MAIN FUNCTION
# ==================================================
# Controls the complete preprocessing pipeline.

def main():
    """
    Main function that loads the TinyStories dataset,
    tokenizes every story using the GPT-2 tokenizer,
    and saves the processed files for model training.
    """

    # Create the output directory if it does not already exist.
    # exist_ok=True prevents an error if the folder already exists.
    os.makedirs(
        OUTPUT_DIR,
        exist_ok=True
    )

    # ==================================================
    # STEP 1 : LOAD DATASET
    # ==================================================

    # Display a message indicating that dataset loading has started.
    print("Loading TinyStories dataset...")

    # Search for all training Parquet files inside DATASET_PATH.
    # Example:
    # train-00000.parquet
    # train-00001.parquet
    train_files = sorted(
        glob.glob(
            os.path.join(
                DATASET_PATH,
                "train*.parquet"
            )
        )
    )

    # Search for all validation Parquet files.
    val_files = sorted(
        glob.glob(
            os.path.join(
                DATASET_PATH,
                "validation*.parquet"
            )
        )
    )

    # Verify that at least one training file exists.
    if len(train_files) == 0:
        raise FileNotFoundError(
            f"No training parquet files found in '{DATASET_PATH}'."
        )

    # Verify that at least one validation file exists.
    if len(val_files) == 0:
        raise FileNotFoundError(
            f"No validation parquet files found in '{DATASET_PATH}'."
        )

    # Display the number of training files found.
    print(f"Found {len(train_files)} training files.")

    # Display the number of validation files found.
    print(f"Found {len(val_files)} validation files.")

    # --------------------------------------------------
    # Load Training Dataset
    # --------------------------------------------------

    # Read every training Parquet file into a DataFrame.
    # tqdm displays a progress bar while loading.
    train_dataset = pd.concat(
        [
            # Read one Parquet file.
            pd.read_parquet(f)

            # Loop through every training file.
            for f in tqdm(
                train_files,
                desc="Loading Training Files"
            )
        ],

        # Create new row indices after concatenation.
        ignore_index=True
    )

    # --------------------------------------------------
    # Load Validation Dataset
    # --------------------------------------------------

    # Read every validation Parquet file.
    val_dataset = pd.concat(
        [
            # Read one validation file.
            pd.read_parquet(f)

            # Loop through every validation file.
            for f in tqdm(
                val_files,
                desc="Loading Validation Files"
            )
        ],

        # Reset row indices after concatenation.
        ignore_index=True
    )

    # Display the number of training samples.
    print(
        f"Training samples: {len(train_dataset):,}"
    )

    # Display the number of validation samples.
    print(
        f"Validation samples: {len(val_dataset):,}"
    )

    # ==================================================
    # STEP 2 : INITIALIZE TOKENIZER
    # ==================================================

    # Inform the user that the GPT-2 tokenizer
    # is being initialized.
    print("Initializing GPT-2 tokenizer...")

    # Load the GPT-2 Byte Pair Encoding tokenizer.
    enc = tiktoken.get_encoding(
        TOKENIZER_NAME
    )

    # Obtain the tokenizer vocabulary size.
    vocab_size = enc.n_vocab

    # Display the vocabulary size.
    print(
        f"Vocabulary Size: {vocab_size:,}"
    )

    # ==================================================
    # STEP 3 : TOKENIZE DATASET
    # ==================================================

    # Convert every training story into GPT token IDs.
    train_ids = encode_corpus(

        # Extract the "text" column,
        # convert every entry to string,
        # then convert to a Python list.
        train_dataset["text"].astype(str).tolist(),

        # Pass the tokenizer object.
        enc
    )

    # Convert every validation story into GPT token IDs.
    val_ids = encode_corpus(

        # Extract validation text.
        val_dataset["text"].astype(str).tolist(),

        # Pass the tokenizer.
        enc
    )

    # ==================================================
    # STEP 4 : SAVE TOKEN FILES
    # ==================================================

    # Save training token IDs into train.bin.
    write_memmap(

        # Output file path.
        os.path.join(
            OUTPUT_DIR,
            "train.bin"
        ),

        # Training token IDs.
        train_ids
    )

    # Save validation token IDs into val.bin.
    write_memmap(

        # Output file path.
        os.path.join(
            OUTPUT_DIR,
            "val.bin"
        ),

        # Validation token IDs.
        val_ids
    )

    # ==================================================
    # STEP 5 : SAVE METADATA
    # ==================================================

    # Create a dictionary containing information
    # required during model training.
    meta = {

        # Store tokenizer name.
        "tokenizer_name": TOKENIZER_NAME,

        # Store tokenizer vocabulary size.
        "vocab_size": vocab_size
    }

    # Open (or create) the metadata JSON file.
    with open(

        # Metadata file path.
        os.path.join(
            OUTPUT_DIR,
            "meta.json"
        ),

        # Open in write mode.
        "w"

    ) as f:

        # Save the dictionary as a formatted JSON file.
        json.dump(
            meta,
            f,
            indent=4
        )

    # Display a completion message.
    print("\n====================================")
    print("Dataset preprocessing completed.")
    print(f"Files saved to: {OUTPUT_DIR}")
    print("====================================")